In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/raw/interventional_diabetes_trials_raw.csv")

print(df[['start_date', 'completion_date']].head())

In [ ]:
# Clean intervention_type
import ast

def extract_intervention_type(x):
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except:
            return None

    if isinstance(x, list) and len(x) > 0:
        first = x[0]
        if isinstance(first, dict):
            return first.get("type")
    return None

df['intervention_type'] = df['intervention_type'].apply(extract_intervention_type)

df['intervention_type'] = df['intervention_type'].fillna("UNKNOWN")

# Check result
print(df['intervention_type'].value_counts())

In [ ]:
print(df['intervention_type'].iloc[0])

In [ ]:
df[['intervention_type', 'phase']].head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
#Convert Dates
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['completion_date'] = pd.to_datetime(df['completion_date'], errors='coerce')

#Check if conversion worked
df[['start_date', 'completion_date']].head()

In [ ]:
#Create Study Duration
df['study_duration_days'] = (df['completion_date'] - df['start_date']).dt.days

# Fill missing durations ONLY where start_date exists
mask = df['study_duration_days'].isna() & df['start_date'].notna()

df.loc[mask, 'study_duration_days'] = (
    pd.Timestamp.today() - df.loc[mask, 'start_date']
).dt.days

df['study_duration_days'] = df['study_duration_days'].fillna(df['study_duration_days'].median())

#Check column
df['study_duration_days'].isna().sum()

In [ ]:
#Clean Enrollment
df['enrollment'] = pd.to_numeric(df['enrollment'], errors='coerce')

In [ ]:
#Fill missing enrollment with median
df['enrollment'] = df['enrollment'].fillna(df['enrollment'].median())

In [ ]:
#Clean locations (Sites)
df['locations'] = pd.to_numeric(df['locations'], errors='coerce')
df['locations'] = df['locations'].fillna(1) # Assuming missing locations means at least 1 site

In [ ]:
#Extract first phase if multiple phases are listed
df['phase'] = df['phase'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x
)

#Handle missing/NA phases
df['phase'] = df['phase'].replace("NA", "NOT_APPLICABLE")
df['phase'] = df['phase'].fillna("UNKNOWN")

In [ ]:
#Check final cleaned data
print(df.shape)

print(df['status'].value_counts())
print(df['phase'].value_counts())
print(df['study_duration_days'].describe())

In [ ]:
#Save cleaned data
import os
os.makedirs("../data/processed/", exist_ok=True)
df.to_csv("../data/processed/clean_trials.csv", index=False)

print("Clean data saved to data/processed/")